# Notebook 02 — PPMI Cohort Definition and Outcome Construction

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using Publicly Available Multimodal Data  
**Dataset:** Parkinson’s Progression Markers Initiative (PPMI)  
**Notebook stage:** Cohort definition and motor progression outcome construction  
**Primary outcome candidate:** Longitudinal change in MDS-UPDRS Part III total score (`NP3TOT`)

---

## Objective

This notebook uses the verified PPMI clinical files from Notebook 01 to:

1. Define the primary analytic cohort: **Parkinson’s disease participants only**.
2. Clean duplicate MDS-UPDRS Part III rows caused by repeated motor examination states.
3. Select one MDS-UPDRS Part III record per participant per visit using an explicit and reproducible exam-state rule.
4. Construct candidate motor progression outcomes at follow-up windows such as `V04`, `V06`, and `V08`.
5. Calculate:
   - `delta_NP3TOT = follow-up NP3TOT − baseline NP3TOT`
   - `annualized_delta_NP3TOT`
   - data-driven rapid progression labels using the upper quartile of annualized change.
6. Save clean outcome datasets and QC reports.

---

## Important stage rule

This notebook **does not run machine learning**.  
It only creates a clean, auditable analytic cohort and candidate outcome definitions.

## Scientific Background

Parkinson’s disease is progressive, so a clinically useful machine learning project should focus on predicting **future progression**, not only classifying Parkinson’s disease versus controls.

For this project, the primary candidate outcome is change in **MDS-UPDRS Part III**, which measures motor examination severity. Because PPMI participants may have repeated Part III assessments in different medication states, the outcome must be constructed carefully before any model development.

---

## Key methodological issue

The same participant and visit can have more than one Part III row, commonly reflecting different motor examination states, for example:

- OFF medication examination
- ON medication examination
- unmedicated or standard examination

Therefore, we must select one row per `PATNO` and `EVENT_ID` using a transparent rule before calculating motor progression.

## Dataset Verification

This notebook expects that Notebook 01 has already been run successfully and that the project folder has this structure in Google Drive:

```text
MyDrive/
└── PPMI_PD_Progression/
    ├── data/
    │   ├── raw/
    │   └── extracted/
    └── outputs/
        └── notebook_01_variable_inventory/
```

Required files from the previous stage include:

- `Participant_Status_30Jun2026.csv`
- `MDS-UPDRS_Part_III_30Jun2026.csv`

Recommended additional files for baseline covariate availability:

- MDS-UPDRS Part I
- MDS-UPDRS Part I Patient Questionnaire
- MDS-UPDRS Part II Patient Questionnaire
- MoCA
- UPSIT
- Vital Signs
- PD Diagnosis History
- Primary Research Diagnosis

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Google Drive mount step skipped or not running in Colab.")
    print("Details:", e)

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import zipfile
import warnings
import re
import json

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")
RAW_DIR = PROJECT_DIR / "data" / "raw"
EXTRACT_DIR = PROJECT_DIR / "data" / "extracted"
NOTEBOOK01_OUT = PROJECT_DIR / "outputs" / "notebook_01_variable_inventory"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "notebook_02_cohort_outcome"

for d in [RAW_DIR, EXTRACT_DIR, NOTEBOOK01_OUT, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR exists:", RAW_DIR.exists())
print("EXTRACT_DIR exists:", EXTRACT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 03. Extract ZIP files if needed
# ============================================================

zip_files = sorted(RAW_DIR.glob("*.zip"))

print(f"ZIP files found in raw folder: {len(zip_files)}")
for z in zip_files:
    print("-", z.name)

for z in zip_files:
    target_subdir = EXTRACT_DIR / z.stem
    target_subdir.mkdir(parents=True, exist_ok=True)
    existing_csvs = list(target_subdir.rglob("*.csv"))
    if len(existing_csvs) == 0:
        print(f"Extracting: {z.name}")
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(target_subdir)
    else:
        print(f"Already extracted: {z.name}")

In [ ]:
# ============================================================
# 04. Locate CSV files
# ============================================================

csv_files = sorted(set(list(RAW_DIR.rglob("*.csv")) + list(EXTRACT_DIR.rglob("*.csv"))))

print(f"Total CSV files found: {len(csv_files)}")
for p in csv_files:
    try:
        rel = p.relative_to(PROJECT_DIR)
    except Exception:
        rel = p
    print("-", rel)

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV files found. Confirm that raw ZIP/CSV files are in: "
        f"{RAW_DIR}"
    )

In [ ]:
# ============================================================
# 05. Helper functions
# ============================================================

def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)

def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

def find_csv_by_keywords(include_keywords, exclude_keywords=None, required=True):
    '''
    Find one CSV file by keywords in the filename.
    include_keywords and exclude_keywords are case-insensitive substrings.
    '''
    exclude_keywords = exclude_keywords or []
    candidates = []
    for p in csv_files:
        n = normalize_name(p.name)
        include_ok = all(k.lower() in n for k in include_keywords)
        exclude_ok = not any(k.lower() in n for k in exclude_keywords)
        if include_ok and exclude_ok:
            candidates.append(p)

    if len(candidates) == 0:
        if required:
            raise FileNotFoundError(f"No file matched include={include_keywords}, exclude={exclude_keywords}")
        return None

    # Prefer shorter filenames and raw/extracted consistency
    candidates = sorted(candidates, key=lambda x: (len(x.name), x.name))
    return candidates[0]

def standardize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

def parse_month_year(s):
    return pd.to_datetime(s, format="%m/%Y", errors="coerce")

def yes_no_flag(series):
    return series.map({1: 1, 0: 0, "1": 1, "0": 0, "YES": 1, "NO": 0, "Yes": 1, "No": 0})

def print_shape(name, df):
    print(f"{name}: rows={df.shape[0]:,}, columns={df.shape[1]:,}")

In [ ]:
# ============================================================
# 06. Load required and optional datasets
# ============================================================

file_map = {}

file_map["participant_status"] = find_csv_by_keywords(["participant", "status"], required=True)
file_map["mds_updrs_part_iii"] = find_csv_by_keywords(["part_iii"], required=True)

# Optional baseline predictor datasets
optional_specs = {
    "mds_updrs_part_i": (["part_i"], ["patient", "iii", "iv", "ii"]),
    "mds_updrs_part_i_patient": (["part_i_patient"], ["iii", "iv", "ii"]),
    "mds_updrs_part_ii": (["part_ii"], ["iii", "iv"]),
    "mds_updrs_part_iv": (["part_iv"], ["iii", "ii"]),
    "moca": (["moca"], []),
    "upsit": (["upsit"], []),
    "vital_signs": (["vital"], []),
    "pd_diagnosis_history": (["pd_diagnosis"], []),
    "primary_research_diagnosis": (["primary_research"], []),
}

for key, (inc, exc) in optional_specs.items():
    try:
        file_map[key] = find_csv_by_keywords(inc, exc, required=False)
    except Exception:
        file_map[key] = None

print("Resolved file map:")
for k, v in file_map.items():
    print(f"{k}: {v.name if v else 'NOT FOUND'}")

datasets = {}
for key, path in file_map.items():
    if path is not None:
        datasets[key] = standardize_columns(read_csv_safely(path))
        print_shape(key, datasets[key])

In [ ]:
# ============================================================
# 07. Required variable checks
# ============================================================

ps = datasets["participant_status"].copy()
part3 = datasets["mds_updrs_part_iii"].copy()

required_ps_cols = ["PATNO", "COHORT", "COHORT_DEFINITION"]
required_part3_cols = ["PATNO", "EVENT_ID", "NP3TOT"]

for c in required_ps_cols:
    if c not in ps.columns:
        raise ValueError(f"Participant Status is missing required column: {c}")

for c in required_part3_cols:
    if c not in part3.columns:
        raise ValueError(f"MDS-UPDRS Part III is missing required column: {c}")

ps["PATNO"] = numeric_series(ps["PATNO"]).astype("Int64")
part3["PATNO"] = numeric_series(part3["PATNO"]).astype("Int64")
part3["NP3TOT"] = numeric_series(part3["NP3TOT"])

print("Required checks passed.")
print("Participant Status unique PATNO:", ps["PATNO"].nunique())
print("Part III unique PATNO:", part3["PATNO"].nunique())
print("Part III rows with non-missing NP3TOT:", part3["NP3TOT"].notna().sum())

## Cohort Definition

Primary cohort for this project:

```text
COHORT = 1
COHORT_DEFINITION = Parkinson's Disease
```

Healthy controls, SWEDD, and prodromal participants are not included in the primary progression model. They may be used later for secondary or comparative analyses, but not in the primary disease-progression cohort.

In [ ]:
# ============================================================
# 08. Define Parkinson's disease cohort
# ============================================================

ps["COHORT_DEFINITION_STR"] = ps["COHORT_DEFINITION"].astype(str)

pd_mask = (
    (numeric_series(ps["COHORT"]) == 1) |
    (ps["COHORT_DEFINITION_STR"].str.contains("Parkinson", case=False, na=False))
)

pd_participants = ps.loc[pd_mask].copy()
pd_patnos = set(pd_participants["PATNO"].dropna().astype(int).tolist())

cohort_distribution = (
    ps.groupby(["COHORT", "COHORT_DEFINITION"], dropna=False)
      .agg(n_participants=("PATNO", "nunique"))
      .reset_index()
      .sort_values("COHORT")
)

cohort_distribution.to_csv(OUTPUT_DIR / "01_cohort_distribution.csv", index=False)

print(cohort_distribution.to_string(index=False))
print("\nPrimary PD cohort size:", len(pd_patnos))

## MDS-UPDRS Part III duplicate handling

The rule below creates one selected Part III record per participant per visit.

### Exam-state classification

Rows are classified as:

- `OFF`: explicit OFF medication examination
- `ON`: explicit ON medication examination
- `UNMEDICATED_STANDARD`: participant appears untreated/unmedicated at the visit
- `STANDARD_UNSPECIFIED`: standard Part III exam without clear medication state
- `UNKNOWN`: insufficient information

### Selection priority

For each `PATNO` and `EVENT_ID`, the selected row is chosen using this order:

1. `OFF`
2. `UNMEDICATED_STANDARD`
3. `STANDARD_UNSPECIFIED`
4. `ON`
5. `UNKNOWN`

The `ON` row is not preferred for the primary outcome. If an ON-only row is selected because no better row exists, it is flagged and can be excluded from the strict primary analysis.

In [ ]:
# ============================================================
# 09. Classify MDS-UPDRS Part III motor examination state
# ============================================================

def is_one(x):
    try:
        return float(x) == 1.0
    except Exception:
        return False

def is_zero(x):
    try:
        return float(x) == 0.0
    except Exception:
        return False

def classify_part3_exam_state(row):
    pdstate = str(row.get("PDSTATE", "")).strip().upper()
    page = str(row.get("PAG_NAME", "")).strip().upper()

    # Explicit OFF examination
    if pdstate == "OFF" or is_one(row.get("OFFEXAM")) or page in ["NUPDR3OF"]:
        return "OFF"

    # Explicit ON examination
    if pdstate == "ON" or is_one(row.get("ONEXAM")) or page in ["NUPDRS3A", "NUPDR3ON"]:
        return "ON"

    # Untreated / standard exam
    if is_zero(row.get("PDMEDYN")) or is_zero(row.get("PDTRTMNT")):
        return "UNMEDICATED_STANDARD"

    # Older or standard exam without clear state
    if page == "NUPDRS3":
        return "STANDARD_UNSPECIFIED"

    return "UNKNOWN"

exam_priority = {
    "OFF": 1,
    "UNMEDICATED_STANDARD": 2,
    "STANDARD_UNSPECIFIED": 3,
    "ON": 4,
    "UNKNOWN": 5
}

part3_work = part3.copy()
part3_work["exam_state"] = part3_work.apply(classify_part3_exam_state, axis=1)
part3_work["exam_priority"] = part3_work["exam_state"].map(exam_priority).fillna(99).astype(int)

# Dates used only for deterministic sorting and time-difference estimation
part3_work["EXAMDT_parsed"] = parse_month_year(part3_work["EXAMDT"]) if "EXAMDT" in part3_work.columns else pd.NaT
part3_work["INFODT_parsed"] = parse_month_year(part3_work["INFODT"]) if "INFODT" in part3_work.columns else pd.NaT
part3_work["event_date"] = part3_work["EXAMDT_parsed"].fillna(part3_work["INFODT_parsed"])

part3_work["REC_ID_sort"] = pd.to_numeric(part3_work.get("REC_ID"), errors="coerce")

state_distribution_before = (
    part3_work.assign(has_np3tot=part3_work["NP3TOT"].notna())
              .groupby(["exam_state", "has_np3tot"], dropna=False)
              .size()
              .reset_index(name="n_rows")
              .sort_values(["exam_state", "has_np3tot"])
)

state_distribution_before.to_csv(OUTPUT_DIR / "02_part3_exam_state_distribution_before_dedup.csv", index=False)
print(state_distribution_before.to_string(index=False))

In [ ]:
# ============================================================
# 10. Deduplicate MDS-UPDRS Part III
# ============================================================

part3_nonmissing = part3_work.loc[part3_work["NP3TOT"].notna()].copy()

duplicates_before = int(part3_nonmissing.duplicated(["PATNO", "EVENT_ID"]).sum())

part3_selected = (
    part3_nonmissing
    .sort_values(
        by=["PATNO", "EVENT_ID", "exam_priority", "event_date", "REC_ID_sort"],
        ascending=[True, True, True, True, True],
        na_position="last"
    )
    .drop_duplicates(["PATNO", "EVENT_ID"], keep="first")
    .copy()
)

duplicates_after = int(part3_selected.duplicated(["PATNO", "EVENT_ID"]).sum())

dedup_summary = pd.DataFrame([
    {"metric": "part3_rows_total", "value": len(part3_work)},
    {"metric": "part3_rows_with_nonmissing_np3tot", "value": len(part3_nonmissing)},
    {"metric": "duplicate_PATNO_EVENT_ID_before_dedup", "value": duplicates_before},
    {"metric": "selected_rows_after_dedup", "value": len(part3_selected)},
    {"metric": "duplicate_PATNO_EVENT_ID_after_dedup", "value": duplicates_after},
])

dedup_state_distribution = (
    part3_selected
    .groupby("exam_state", dropna=False)
    .agg(n_selected_rows=("PATNO", "size"),
         n_unique_participants=("PATNO", "nunique"))
    .reset_index()
    .sort_values("n_selected_rows", ascending=False)
)

dedup_summary.to_csv(OUTPUT_DIR / "03_part3_deduplication_summary.csv", index=False)
dedup_state_distribution.to_csv(OUTPUT_DIR / "04_part3_exam_state_distribution_after_dedup.csv", index=False)
part3_selected.to_csv(OUTPUT_DIR / "05_mds_updrs_part3_selected_one_row_per_visit.csv", index=False)

print(dedup_summary.to_string(index=False))
print("\nSelected exam-state distribution:")
print(dedup_state_distribution.to_string(index=False))

## Outcome Construction

Candidate outcome:

```text
delta_NP3TOT = follow-up NP3TOT − baseline NP3TOT
```

The notebook creates candidate datasets for several follow-up visits. The primary recommendation will be based on a balance between:

- sufficient sample size,
- longer follow-up duration,
- availability of non-ON or OFF/unmedicated Part III examinations,
- transparent avoidance of medication-state contamination.

The current default candidate windows are:

```text
V04, V06, V08, V10, V12
```

In [ ]:
# ============================================================
# 11. Construct candidate motor progression outcome datasets
# ============================================================

BASELINE_EVENT = "BL"
CANDIDATE_FOLLOWUP_EVENTS = ["V04", "V06", "V08", "V10", "V12"]

# Fallback year assumptions are only used when dates are missing.
# Actual month/year differences from INFODT/EXAMDT are preferred whenever available.
VISIT_YEAR_FALLBACK = {
    "V04": 1.0,
    "V06": 2.0,
    "V08": 3.0,
    "V10": 4.0,
    "V12": 5.0,
    "V14": 6.0,
    "V17": 8.0,
}

PRIMARY_ALLOWED_STATES = ["OFF", "UNMEDICATED_STANDARD"]
SENSITIVITY_ALLOWED_STATES = ["OFF", "UNMEDICATED_STANDARD", "STANDARD_UNSPECIFIED"]

part3_selected_pd = part3_selected.loc[part3_selected["PATNO"].astype("Int64").isin(pd_patnos)].copy()

baseline_cols = [
    "PATNO", "EVENT_ID", "NP3TOT", "exam_state", "exam_priority",
    "PAG_NAME", "PDSTATE", "PDMEDYN", "PDTRTMNT", "event_date", "INFODT", "EXAMDT"
]
baseline_cols = [c for c in baseline_cols if c in part3_selected_pd.columns]

baseline = (
    part3_selected_pd.loc[part3_selected_pd["EVENT_ID"].eq(BASELINE_EVENT), baseline_cols]
    .copy()
    .rename(columns={
        "EVENT_ID": "baseline_event",
        "NP3TOT": "baseline_NP3TOT",
        "exam_state": "baseline_exam_state",
        "exam_priority": "baseline_exam_priority",
        "PAG_NAME": "baseline_PAG_NAME",
        "PDSTATE": "baseline_PDSTATE",
        "PDMEDYN": "baseline_PDMEDYN",
        "PDTRTMNT": "baseline_PDTRTMNT",
        "event_date": "baseline_date",
        "INFODT": "baseline_INFODT",
        "EXAMDT": "baseline_EXAMDT",
    })
)

candidate_summaries = []
candidate_datasets = {}

for followup_event in CANDIDATE_FOLLOWUP_EVENTS:
    followup = (
        part3_selected_pd.loc[part3_selected_pd["EVENT_ID"].eq(followup_event), baseline_cols]
        .copy()
        .rename(columns={
            "EVENT_ID": "followup_event",
            "NP3TOT": "followup_NP3TOT",
            "exam_state": "followup_exam_state",
            "exam_priority": "followup_exam_priority",
            "PAG_NAME": "followup_PAG_NAME",
            "PDSTATE": "followup_PDSTATE",
            "PDMEDYN": "followup_PDMEDYN",
            "PDTRTMNT": "followup_PDTRTMNT",
            "event_date": "followup_date",
            "INFODT": "followup_INFODT",
            "EXAMDT": "followup_EXAMDT",
        })
    )

    paired = baseline.merge(followup, on="PATNO", how="inner")
    paired = paired.merge(
        pd_participants[[
            "PATNO", "COHORT", "COHORT_DEFINITION", "ENROLL_AGE",
            "ENRLLRRK2", "ENRLGBA", "ENRLSNCA", "ENRLPRKN", "ENRLRBD", "ENRLHPSM"
        ]].copy(),
        on="PATNO",
        how="left"
    )

    paired["delta_NP3TOT"] = paired["followup_NP3TOT"] - paired["baseline_NP3TOT"]

    date_diff_years = (
        (paired["followup_date"] - paired["baseline_date"]).dt.days / 365.25
        if ("followup_date" in paired.columns and "baseline_date" in paired.columns)
        else np.nan
    )

    paired["followup_years"] = date_diff_years
    paired.loc[paired["followup_years"].isna() | (paired["followup_years"] <= 0), "followup_years"] = VISIT_YEAR_FALLBACK.get(followup_event, np.nan)

    paired["annualized_delta_NP3TOT"] = paired["delta_NP3TOT"] / paired["followup_years"]

    paired["primary_exam_state_eligible"] = (
        paired["baseline_exam_state"].isin(PRIMARY_ALLOWED_STATES) &
        paired["followup_exam_state"].isin(PRIMARY_ALLOWED_STATES)
    )

    paired["sensitivity_exam_state_eligible"] = (
        paired["baseline_exam_state"].isin(SENSITIVITY_ALLOWED_STATES) &
        paired["followup_exam_state"].isin(SENSITIVITY_ALLOWED_STATES)
    )

    paired["on_exam_in_pair"] = (
        paired["baseline_exam_state"].eq("ON") |
        paired["followup_exam_state"].eq("ON")
    )

    primary_values = paired.loc[
        paired["primary_exam_state_eligible"] & paired["annualized_delta_NP3TOT"].notna(),
        "annualized_delta_NP3TOT"
    ]

    if len(primary_values) > 0:
        q75 = float(primary_values.quantile(0.75))
    else:
        q75 = np.nan

    paired["rapid_progression_q75"] = np.where(
        paired["primary_exam_state_eligible"] & paired["annualized_delta_NP3TOT"].notna(),
        (paired["annualized_delta_NP3TOT"] >= q75).astype(int),
        np.nan
    )

    paired["any_motor_worsening"] = np.where(
        paired["delta_NP3TOT"].notna(),
        (paired["delta_NP3TOT"] > 0).astype(int),
        np.nan
    )

    candidate_datasets[followup_event] = paired

    out_path = OUTPUT_DIR / f"06_outcome_dataset_{followup_event}.csv"
    paired.to_csv(out_path, index=False)

    eligible_primary = paired.loc[paired["primary_exam_state_eligible"]].copy()

    candidate_summaries.append({
        "followup_event": followup_event,
        "n_pairs_any_selected_exam": len(paired),
        "n_primary_exam_state_eligible": int(paired["primary_exam_state_eligible"].sum()),
        "n_sensitivity_exam_state_eligible": int(paired["sensitivity_exam_state_eligible"].sum()),
        "n_pairs_with_ON_exam_flag": int(paired["on_exam_in_pair"].sum()),
        "mean_delta_NP3TOT_primary": float(eligible_primary["delta_NP3TOT"].mean()) if len(eligible_primary) else np.nan,
        "median_delta_NP3TOT_primary": float(eligible_primary["delta_NP3TOT"].median()) if len(eligible_primary) else np.nan,
        "iqr_delta_NP3TOT_primary": (
            f"{eligible_primary['delta_NP3TOT'].quantile(0.25):.2f} to {eligible_primary['delta_NP3TOT'].quantile(0.75):.2f}"
            if len(eligible_primary) else np.nan
        ),
        "q75_annualized_delta_NP3TOT_primary": q75,
        "n_rapid_progressors_q75_primary": int((paired["rapid_progression_q75"] == 1).sum()),
        "median_followup_years_primary": float(eligible_primary["followup_years"].median()) if len(eligible_primary) else np.nan,
    })

candidate_summary = pd.DataFrame(candidate_summaries)
candidate_summary.to_csv(OUTPUT_DIR / "07_candidate_outcome_summary.csv", index=False)

print(candidate_summary.to_string(index=False))

In [ ]:
# ============================================================
# 12. Recommend a primary follow-up window based on sample size and duration
# ============================================================

# Recommendation rule:
# Among V04/V06/V08, prefer the longest follow-up with at least 800 primary eligible pairs.
# If none meet this threshold, choose the candidate with the largest number of primary eligible pairs.

RECOMMENDED_CANDIDATES = ["V04", "V06", "V08"]
MIN_PRIMARY_ELIGIBLE_N = 800

summary_short = candidate_summary[candidate_summary["followup_event"].isin(RECOMMENDED_CANDIDATES)].copy()
eligible_for_rule = summary_short[summary_short["n_primary_exam_state_eligible"] >= MIN_PRIMARY_ELIGIBLE_N].copy()

if len(eligible_for_rule) > 0:
    # Preserve intended follow-up order and select the longest among eligible
    order = {v: i for i, v in enumerate(RECOMMENDED_CANDIDATES)}
    eligible_for_rule["order"] = eligible_for_rule["followup_event"].map(order)
    recommended_event = eligible_for_rule.sort_values("order").iloc[-1]["followup_event"]
    recommendation_reason = (
        f"Selected the longest candidate among V04/V06/V08 with at least "
        f"{MIN_PRIMARY_ELIGIBLE_N} primary eligible pairs."
    )
else:
    recommended_event = summary_short.sort_values("n_primary_exam_state_eligible", ascending=False).iloc[0]["followup_event"]
    recommendation_reason = (
        f"No V04/V06/V08 candidate reached {MIN_PRIMARY_ELIGIBLE_N} primary eligible pairs; "
        "selected the candidate with the largest eligible sample size."
    )

recommendation = pd.DataFrame([{
    "recommended_primary_followup_event": recommended_event,
    "rule": f"Prefer longest V04/V06/V08 with n_primary_exam_state_eligible >= {MIN_PRIMARY_ELIGIBLE_N}",
    "reason": recommendation_reason
}])

recommendation.to_csv(OUTPUT_DIR / "08_recommended_primary_followup_window.csv", index=False)

print(recommendation.to_string(index=False))

## Baseline Predictor Availability

This section creates a participant-level baseline table for PD participants using values from `BL` when available, otherwise `SC` when scientifically reasonable for baseline characterization.

This is still not the final modeling dataset.  
It is a checkpoint to confirm that the core predictors are available before Notebook 03.

In [ ]:
# ============================================================
# 13. Build baseline predictor table
# ============================================================

def get_baseline_value(df, value_cols, dataset_label, event_priority=("BL", "SC")):
    '''
    Return one row per PATNO with baseline values.
    Uses EVENT_ID priority order: BL first, then SC.
    '''
    if df is None or "PATNO" not in df.columns:
        return pd.DataFrame({"PATNO": []})

    df = df.copy()
    df["PATNO"] = numeric_series(df["PATNO"]).astype("Int64")

    if "EVENT_ID" not in df.columns:
        available = [c for c in value_cols if c in df.columns]
        return df[["PATNO"] + available].drop_duplicates("PATNO")

    available = [c for c in value_cols if c in df.columns]
    if len(available) == 0:
        return pd.DataFrame({"PATNO": df["PATNO"].drop_duplicates()})

    df = df.loc[df["EVENT_ID"].isin(event_priority), ["PATNO", "EVENT_ID"] + available].copy()
    priority_map = {ev: i for i, ev in enumerate(event_priority)}
    df["event_priority"] = df["EVENT_ID"].map(priority_map)

    df = (
        df.sort_values(["PATNO", "event_priority"])
          .drop_duplicates("PATNO", keep="first")
          .copy()
    )

    rename_map = {c: f"{dataset_label}_{c}" for c in available}
    df = df.rename(columns=rename_map)
    df = df.rename(columns={"EVENT_ID": f"{dataset_label}_baseline_source_event"})
    keep_cols = ["PATNO", f"{dataset_label}_baseline_source_event"] + list(rename_map.values())
    return df[keep_cols]

baseline_predictors = pd_participants.copy()

# Genetic/subgroup flags from Participant Status are kept as baseline enrollment features.
participant_cols = [
    "PATNO", "COHORT", "COHORT_DEFINITION", "ENROLL_AGE",
    "ENRLLRRK2", "ENRLGBA", "ENRLSNCA", "ENRLPRKN", "ENRLRBD", "ENRLHPSM"
]
participant_cols = [c for c in participant_cols if c in baseline_predictors.columns]
baseline_predictors = baseline_predictors[participant_cols].copy()

# Add selected baseline Part III information
baseline_part3_for_predictors = (
    part3_selected_pd.loc[part3_selected_pd["EVENT_ID"].eq(BASELINE_EVENT),
                          ["PATNO", "NP3TOT", "NHY", "exam_state"] if "NHY" in part3_selected_pd.columns else ["PATNO", "NP3TOT", "exam_state"]]
    .drop_duplicates("PATNO")
    .rename(columns={
        "NP3TOT": "baseline_NP3TOT",
        "NHY": "baseline_NHY",
        "exam_state": "baseline_part3_exam_state"
    })
)
baseline_predictors = baseline_predictors.merge(baseline_part3_for_predictors, on="PATNO", how="left")

# Optional predictors
optional_baseline_specs = [
    ("mds_updrs_part_i", "part1", ["NP1RTOT"]),
    ("mds_updrs_part_i_patient", "part1p", ["NP1PTOT"]),
    ("mds_updrs_part_ii", "part2", ["NP2PTOT"]),
    ("moca", "moca", ["MCATOT"]),
    ("upsit", "upsit", ["TOTAL_CORRECT"]),
    ("vital_signs", "vitals", ["WGTKG", "HTCM", "SYSSUP", "DIASUP", "HRSUP"]),
    ("pd_diagnosis_history", "pddx", ["SXDT", "PDDXDT", "DXTREMOR", "DXRIGID", "DXBRADY", "DXPOSINS", "DOMSIDE"]),
    ("primary_research_diagnosis", "primdiag", ["PRIMDIAG", "DXLVL"]),
]

baseline_availability_rows = []

for dataset_key, label, value_cols in optional_baseline_specs:
    df = datasets.get(dataset_key)
    if df is None:
        baseline_availability_rows.append({
            "dataset": dataset_key,
            "status": "NOT_LOADED",
            "available_variables": "",
            "n_pd_with_any_baseline_value": 0
        })
        continue

    bdf = get_baseline_value(df, value_cols, label)
    baseline_predictors = baseline_predictors.merge(bdf, on="PATNO", how="left")

    available = [f"{label}_{c}" for c in value_cols if f"{label}_{c}" in bdf.columns]
    n_any = int(bdf.loc[bdf["PATNO"].isin(pd_patnos), available].notna().any(axis=1).sum()) if available else 0

    baseline_availability_rows.append({
        "dataset": dataset_key,
        "status": "LOADED",
        "available_variables": ", ".join(available),
        "n_pd_with_any_baseline_value": n_any
    })

baseline_predictors.to_csv(OUTPUT_DIR / "09_baseline_predictor_table_PD.csv", index=False)
baseline_availability = pd.DataFrame(baseline_availability_rows)
baseline_availability.to_csv(OUTPUT_DIR / "10_baseline_predictor_availability.csv", index=False)

print("Baseline predictor table shape:", baseline_predictors.shape)
print(baseline_availability.to_string(index=False))

In [ ]:
# ============================================================
# 14. Create recommended primary analytic cohort file
# ============================================================

recommended_dataset = candidate_datasets[str(recommended_event)].copy()

primary_analytic = (
    recommended_dataset
    .loc[recommended_dataset["primary_exam_state_eligible"]]
    .merge(baseline_predictors, on="PATNO", how="left", suffixes=("", "_baseline_table"))
    .copy()
)

primary_analytic.to_csv(OUTPUT_DIR / "11_primary_analytic_cohort_recommended_window.csv", index=False)

# Keep a sensitivity file too
sensitivity_analytic = (
    recommended_dataset
    .loc[recommended_dataset["sensitivity_exam_state_eligible"]]
    .merge(baseline_predictors, on="PATNO", how="left", suffixes=("", "_baseline_table"))
    .copy()
)

sensitivity_analytic.to_csv(OUTPUT_DIR / "12_sensitivity_analytic_cohort_recommended_window.csv", index=False)

print("Recommended event:", recommended_event)
print("Primary analytic cohort shape:", primary_analytic.shape)
print("Sensitivity analytic cohort shape:", sensitivity_analytic.shape)

print("\nOutcome columns preview:")
preview_cols = [
    "PATNO", "baseline_event", "followup_event", "baseline_NP3TOT", "followup_NP3TOT",
    "delta_NP3TOT", "followup_years", "annualized_delta_NP3TOT",
    "rapid_progression_q75", "baseline_exam_state", "followup_exam_state"
]
preview_cols = [c for c in preview_cols if c in primary_analytic.columns]
print(primary_analytic[preview_cols].head().to_string(index=False))

## Quality Control Checklist

This checklist must be reviewed before moving to Notebook 03.

Notebook 03 should only start if:

1. Part III duplicates were reduced to one row per participant per visit.
2. The primary analytic cohort has enough eligible PD participants.
3. The recommended follow-up window is scientifically acceptable.
4. ON-medication motor examinations are not used in the strict primary outcome.
5. The rapid progression label is created only after the outcome is calculated.
6. No machine learning has been performed in Notebook 02.

In [ ]:
# ============================================================
# 15. QC checklist
# ============================================================

qc_rows = []

qc_rows.append({
    "qc_item": "Participant Status loaded",
    "status": "PASS" if "participant_status" in datasets else "FAIL",
    "detail": f"Rows: {ps.shape[0]:,}"
})

qc_rows.append({
    "qc_item": "Primary PD cohort defined",
    "status": "PASS" if len(pd_patnos) > 0 else "FAIL",
    "detail": f"PD participants: {len(pd_patnos):,}"
})

qc_rows.append({
    "qc_item": "MDS-UPDRS Part III loaded",
    "status": "PASS" if "mds_updrs_part_iii" in datasets else "FAIL",
    "detail": f"Rows: {part3.shape[0]:,}"
})

qc_rows.append({
    "qc_item": "NP3TOT available",
    "status": "PASS" if part3["NP3TOT"].notna().sum() > 0 else "FAIL",
    "detail": f"Rows with non-missing NP3TOT: {part3['NP3TOT'].notna().sum():,}"
})

qc_rows.append({
    "qc_item": "Part III duplicate handling",
    "status": "PASS" if duplicates_after == 0 else "FAIL",
    "detail": f"Before: {duplicates_before:,}; after: {duplicates_after:,}"
})

qc_rows.append({
    "qc_item": "Primary analytic cohort size",
    "status": "PASS" if primary_analytic.shape[0] >= 500 else "WARNING",
    "detail": f"Recommended event {recommended_event}; n={primary_analytic.shape[0]:,}"
})

qc_rows.append({
    "qc_item": "ON examinations excluded from strict primary outcome",
    "status": "PASS" if not primary_analytic[["baseline_exam_state", "followup_exam_state"]].isin(["ON"]).any().any() else "FAIL",
    "detail": "Strict primary analytic cohort uses OFF or UNMEDICATED_STANDARD only."
})

qc_rows.append({
    "qc_item": "Rapid progression label created",
    "status": "PASS" if "rapid_progression_q75" in primary_analytic.columns else "FAIL",
    "detail": "Defined using upper quartile of annualized delta_NP3TOT within the primary eligible cohort."
})

qc_rows.append({
    "qc_item": "No ML modeling performed",
    "status": "PASS",
    "detail": "This notebook constructs cohort and outcome only."
})

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUTPUT_DIR / "13_quality_control_checklist.csv", index=False)

print(qc.to_string(index=False))

In [ ]:
# ============================================================
# 16. Summary report
# ============================================================

report_lines = []

report_lines.append("Notebook 02 — PPMI Cohort Definition and Outcome Construction")
report_lines.append("=" * 72)
report_lines.append("")
report_lines.append(f"Primary cohort: Parkinson's disease participants only")
report_lines.append(f"PD participants in Participant Status: {len(pd_patnos):,}")
report_lines.append("")
report_lines.append("MDS-UPDRS Part III duplicate handling")
report_lines.append(f"- Rows with non-missing NP3TOT before selection: {len(part3_nonmissing):,}")
report_lines.append(f"- Duplicate PATNO/EVENT_ID before selection: {duplicates_before:,}")
report_lines.append(f"- Selected rows after deduplication: {len(part3_selected):,}")
report_lines.append(f"- Duplicate PATNO/EVENT_ID after selection: {duplicates_after:,}")
report_lines.append("")
report_lines.append("Candidate outcome summary")
report_lines.append(candidate_summary.to_string(index=False))
report_lines.append("")
report_lines.append("Recommended primary follow-up window")
report_lines.append(f"- Recommended event: {recommended_event}")
report_lines.append(f"- Reason: {recommendation_reason}")
report_lines.append("")
report_lines.append("Primary analytic cohort")
report_lines.append(f"- Rows: {primary_analytic.shape[0]:,}")
report_lines.append(f"- Columns: {primary_analytic.shape[1]:,}")
report_lines.append("")
report_lines.append("Interpretation")
report_lines.append(
    "The primary outcome dataset is now ready for scientific review. "
    "Machine learning should not begin until the outcome window and rapid progression definition are approved."
)
report_lines.append("")
report_lines.append("Output files are saved in:")
report_lines.append(str(OUTPUT_DIR))

report = "\n".join(report_lines)

with open(OUTPUT_DIR / "14_notebook_02_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print(report)

## Expected Output

This notebook saves the following outputs:

```text
01_cohort_distribution.csv
02_part3_exam_state_distribution_before_dedup.csv
03_part3_deduplication_summary.csv
04_part3_exam_state_distribution_after_dedup.csv
05_mds_updrs_part3_selected_one_row_per_visit.csv
06_outcome_dataset_V04.csv
06_outcome_dataset_V06.csv
06_outcome_dataset_V08.csv
06_outcome_dataset_V10.csv
06_outcome_dataset_V12.csv
07_candidate_outcome_summary.csv
08_recommended_primary_followup_window.csv
09_baseline_predictor_table_PD.csv
10_baseline_predictor_availability.csv
11_primary_analytic_cohort_recommended_window.csv
12_sensitivity_analytic_cohort_recommended_window.csv
13_quality_control_checklist.csv
14_notebook_02_summary_report.txt
```

---

## Stage gate before Notebook 03

Before moving to preprocessing and feature engineering, review:

1. `07_candidate_outcome_summary.csv`
2. `08_recommended_primary_followup_window.csv`
3. `11_primary_analytic_cohort_recommended_window.csv`
4. `13_quality_control_checklist.csv`
5. `14_notebook_02_summary_report.txt`

Only after these are accepted should Notebook 03 begin.